For telco:    
best_model_info
↓
Best Model
↓
Run ID
↓
ROC AUC
↓
Accuracy

For NLP:
best_model_info
↓
Best Model
↓
Run ID
↓
Vectorizer
↓
Vocabulary Size
↓
F1 Macro

In [0]:
# for binary classification (0/1) roc_auc is the best metric
# for multi class classification instead of roc_auc, f1_macro is recommneded for choosing the best model
# Why not accuracy? Accuracy can look good even if one class performs poorly. f1_macro gives equal importance to all classes: Technical, Login, Billing, Cancellation.

#Read model_training_results
results_df = spark.table("dbw_agentic_ai_dev.support_ticket_ai.model_training_results")

#Sort by best metric
display(results_df.orderBy("f1_macro", ascending=False))

#Pick best row 
best_row = (
    results_df
    .orderBy(
        results_df["f1_macro"].desc(),
        results_df["accuracy"].desc()
    )
    .limit(1)
    .collect()[0]
)

best_model_info = [{
    "best_run_id": best_row["run_id"],
    "best_model_name": best_row["model"],
    "best_vectorizer": best_row["vectorizer"],
    "best_vocabulary_size": int(best_row["vocabulary_size"]),
    "best_accuracy": float(best_row["accuracy"]),
    "best_precision_macro": float(best_row["precision_macro"]),
    "best_recall_macro": float(best_row["recall_macro"]),
    "best_f1_macro": float(best_row["f1_macro"])
}]

#Create best_model_info
best_model_df = spark.createDataFrame(best_model_info)

#Write to the delta table
best_model_df.write.format("delta").mode("overwrite").saveAsTable("dbw_agentic_ai_dev.support_ticket_ai.best_model_info")

display(best_model_df)